# 使用 Pinecone 进行嵌入搜索

本 notebook 将带你走完一个简单的流程：下载一些数据、对其进行嵌入，然后使用选定的向量数据库进行索引和搜索。对于希望在安全环境中用自有数据存储和搜索我们的嵌入、以支持聊天机器人、主题建模等生产用例的客户来说，这是一个常见需求。

### 什么是向量数据库

向量数据库是专门用于存储、管理和搜索嵌入向量的数据库。近年来，由于 AI 在解决涉及自然语言、图像识别和其他非结构化数据形式的用例方面越来越有效，使用嵌入将非结构化数据（文本、音频、视频等）编码为向量、供机器学习模型使用的做法呈爆发式增长。向量数据库已成为企业交付和扩展这些用例的有效解决方案。

### 为什么使用向量数据库

向量数据库使企业能够将我们在本仓库中分享的许多嵌入用例（例如问答、聊天机器人和推荐服务）应用于安全、可扩展的环境中。我们的许多客户能在小规模上用嵌入解决问题，但性能和安全性阻碍了他们进入生产环境。我们认为向量数据库是解决这一问题的关键组件。在本指南中，我们将介绍嵌入文本数据、将其存储到向量数据库中并用于语义搜索的基础知识。


### 演示流程
演示流程如下：
- **设置**：导入软件包并设置所需的变量
- **加载数据**：加载数据集并使用 OpenAI embeddings 对其进行嵌入
- **Pinecone**
    - *设置*：这里我们将设置 Pinecone 的 Python 客户端。更多详情请参阅[这里](https://docs.pinecone.io/docs/quickstart)
    - *索引数据*：我们将创建一个带有 __titles__ 和 __content__ 命名空间的索引
    - *搜索数据*：我们将用搜索查询测试这两个命名空间，以确认其正常工作

完成本 notebook 后，你应该会对如何设置和使用向量数据库有基本的了解，并可以继续探索使用我们的嵌入的更复杂用例。

## 设置

导入所需的库，并设置我们要使用的嵌入模型。

In [16]:
# We'll need to install the Pinecone client
!pip install pinecone-client

#Install wget to pull zip file
!pip install wget

In [1]:
import openai

from typing import List, Iterator
import pandas as pd
import numpy as np
import os
import wget
from ast import literal_eval

# Pinecone's client library for Python
import pinecone

# I've set this to our new embeddings model, this can be changed to the embedding model of your choice
EMBEDDING_MODEL = "text-embedding-3-small"

# Ignore unclosed SSL socket warnings - optional in case you get these errors
import warnings

warnings.filterwarnings(action="ignore", message="unclosed", category=ResourceWarning)
warnings.filterwarnings("ignore", category=DeprecationWarning) 

/Users/colin.jarvis/Documents/dev/cookbook/openai-cookbook/vector_db/lib/python3.10/site-packages/pinecone/index.py:4: TqdmExperimentalWarning: Using `tqdm.autonotebook.tqdm` in notebook mode. Use `tqdm.tqdm` instead to force console mode (e.g. in jupyter console)
  from tqdm.autonotebook import tqdm


## 加载数据

在本节中，我们将加载[这篇文章](../../Embedding_Wikipedia_articles_for_search.ipynb)中准备好的嵌入数据。

In [ ]:
embeddings_url = 'https://cdn.openai.com/API/examples/data/vector_database_wikipedia_articles_embedded.zip'

# The file is ~700 MB so this will take some time
wget.download(embeddings_url)

In [ ]:
import zipfile
with zipfile.ZipFile("vector_database_wikipedia_articles_embedded.zip","r") as zip_ref:
    zip_ref.extractall("../data")

In [2]:
article_df = pd.read_csv('../data/vector_database_wikipedia_articles_embedded.csv')

In [3]:
article_df.head()

,id,url,title,text,title_vector,content_vector,vector_id
0,1,https://simple.wikipedia.org/wiki/April,April,April is the fourth month of the year in the J...,"[0.001009464613161981, -0.020700545981526375, ...","[-0.011253940872848034, -0.013491976074874401,...",0
1,2,https://simple.wikipedia.org/wiki/August,August,August (Aug.) is the eighth month of the year ...,"[0.0009286514250561595, 0.000820168002974242, ...","[0.0003609954728744924, 0.007262262050062418, ...",1
2,6,https://simple.wikipedia.org/wiki/Art,Art,Art is a creative activity that expresses imag...,"[0.003393713850528002, 0.0061537534929811954, ...","[-0.004959689453244209, 0.015772193670272827, ...",2
3,8,https://simple.wikipedia.org/wiki/A,A,A or a is the first letter of the English alph...,"[0.0153952119871974, -0.013759135268628597, 0....","[0.024894846603274345, -0.022186409682035446, ...",3
4,9,https://simple.wikipedia.org/wiki/Air,Air,Air refers to the Earth's atmosphere. Air is a...,"[0.02224554680287838, -0.02044147066771984, -0...","[0.021524671465158463, 0.018522677943110466, -...",4


In [4]:
# Read vectors from strings back into a list
article_df['title_vector'] = article_df.title_vector.apply(literal_eval)
article_df['content_vector'] = article_df.content_vector.apply(literal_eval)

# Set vector_id to be a string
article_df['vector_id'] = article_df['vector_id'].apply(str)

In [5]:
article_df.info(show_counts=True)

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 25000 entries, 0 to 24999
Data columns (total 7 columns):
 #   Column          Non-Null Count  Dtype 
---  ------          --------------  ----- 
 0   id              25000 non-null  int64 
 1   url             25000 non-null  object
 2   title           25000 non-null  object
 3   text            25000 non-null  object
 4   title_vector    25000 non-null  object
 5   content_vector  25000 non-null  object
 6   vector_id       25000 non-null  object
dtypes: int64(1), object(6)
memory usage: 1.3+ MB


## Pinecone

接下来我们要看的选项是 **Pinecone**，这是一个提供云原生方案的托管向量数据库。

在进行此步骤之前，你需要前往 [Pinecone](https://www.pinecone.io) 注册，然后将你的 API 密钥保存为名为 ```PINECONE_API_KEY``` 的环境变量。

在本节中，我们将：
- 创建一个包含多个命名空间的索引，分别用于文章标题和内容
- 将数据存储到索引中，并为文章**标题**和**内容**分别设置可搜索的“命名空间”
- 发起一些相似度搜索查询，验证我们的设置是否正常工作

In [6]:
api_key = os.getenv("PINECONE_API_KEY")
pinecone.init(api_key=api_key)

### 创建索引

首先我们需要创建一个索引，命名为 `wikipedia-articles`。有了索引之后，我们就可以创建多个命名空间，让单个索引可以服务于多种搜索用例。更多详情请参阅 [Pinecone 文档](https://docs.pinecone.io/docs/namespaces#:~:text=Pinecone%20allows%20you%20to%20partition,different%20subsets%20of%20your%20index.)。

如果你想并行批量插入索引以提高插入速度，Pinecone 文档中有一篇关于[并行批量插入](https://docs.pinecone.io/docs/insert-data#sending-upserts-in-parallel)的优秀指南。

In [7]:
# Models a simple batch generator that make chunks out of an input DataFrame
class BatchGenerator:
    
    
    def __init__(self, batch_size: int = 10) -> None:
        self.batch_size = batch_size
    
    # Makes chunks out of an input DataFrame
    def to_batches(self, df: pd.DataFrame) -> Iterator[pd.DataFrame]:
        splits = self.splits_num(df.shape[0])
        if splits <= 1:
            yield df
        else:
            for chunk in np.array_split(df, splits):
                yield chunk

    # Determines how many chunks DataFrame contains
    def splits_num(self, elements: int) -> int:
        return round(elements / self.batch_size)
    
    __call__ = to_batches

df_batcher = BatchGenerator(300)

In [8]:
# Pick a name for the new index
index_name = 'wikipedia-articles'

# Check whether the index with the same name already exists - if so, delete it
if index_name in pinecone.list_indexes():
    pinecone.delete_index(index_name)
    
# Creates new index
pinecone.create_index(name=index_name, dimension=len(article_df['content_vector'][0]))
index = pinecone.Index(index_name=index_name)

# Confirm our index was created
pinecone.list_indexes()

['podcasts', 'wikipedia-articles']

In [9]:
# Upsert content vectors in content namespace - this can take a few minutes
print("Uploading vectors to content namespace..")
for batch_df in df_batcher(article_df):
    index.upsert(vectors=zip(batch_df.vector_id, batch_df.content_vector), namespace='content')

Uploading vectors to content namespace..


In [10]:
# Upsert title vectors in title namespace - this can also take a few minutes
print("Uploading vectors to title namespace..")
for batch_df in df_batcher(article_df):
    index.upsert(vectors=zip(batch_df.vector_id, batch_df.title_vector), namespace='title')

Uploading vectors to title namespace..


In [11]:
# Check index size for each namespace to confirm all of our docs have loaded
index.describe_index_stats()

{'dimension': 1536,
 'index_fullness': 0.1,
 'namespaces': {'content': {'vector_count': 25000},
                'title': {'vector_count': 25000}},
 'total_vector_count': 50000}

### 搜索数据

现在我们输入一些示例搜索，检查能否得到不错的结果

In [12]:
# First we'll create dictionaries mapping vector IDs to their outputs so we can retrieve the text for our search results
titles_mapped = dict(zip(article_df.vector_id,article_df.title))
content_mapped = dict(zip(article_df.vector_id,article_df.text))

In [13]:
def query_article(query, namespace, top_k=5):
    '''Queries an article using its title in the specified
     namespace and prints results.'''

    # Create vector embeddings based on the title column
    embedded_query = openai.Embedding.create(
                                            input=query,
                                            model=EMBEDDING_MODEL,
                                            )["data"][0]['embedding']

    # Query namespace passed as parameter using title vector
    query_result = index.query(embedded_query, 
                                      namespace=namespace, 
                                      top_k=top_k)

    # Print query results 
    print(f'\nMost similar results to {query} in "{namespace}" namespace:\n')
    if not query_result.matches:
        print('no query result')
    
    matches = query_result.matches
    ids = [res.id for res in matches]
    scores = [res.score for res in matches]
    df = pd.DataFrame({'id':ids, 
                       'score':scores,
                       'title': [titles_mapped[_id] for _id in ids],
                       'content': [content_mapped[_id] for _id in ids],
                       })
    
    counter = 0
    for k,v in df.iterrows():
        counter += 1
        print(f'{v.title} (score = {v.score})')
    
    print('\n')

    return df

In [14]:
query_output = query_article('modern art in Europe','title')


Most similar results to modern art in Europe in "title" namespace:

Museum of Modern Art (score = 0.875177085)
Western Europe (score = 0.867441177)
Renaissance art (score = 0.864156306)
Pop art (score = 0.860346854)
Northern Europe (score = 0.854658186)




In [15]:
content_query_output = query_article("Famous battles in Scottish history",'content')


Most similar results to Famous battles in Scottish history in "content" namespace:

Battle of Bannockburn (score = 0.869336188)
Wars of Scottish Independence (score = 0.861470938)
1651 (score = 0.852588475)
First War of Scottish Independence (score = 0.84962213)
Robert I of Scotland (score = 0.846214116)


